# Cow Behaviour Classifier

# Imports

In [15]:
import cv2
import tqdm
import os
import pathlib
import random
import shutil
import numpy as np

# CONFIGURATION & CONSTANTS

In [16]:
FRAME_EXTRACTION_INTERVAL: int = 25

MAX_TRAIN_FRAMES = 250
VAL_FRAMES       = 25    
TEST_FRAMES      = 25    

RUMINATE_INTO_STAND_FRAMES = 60
STAND_OWN_FRAMES           = MAX_TRAIN_FRAMES - RUMINATE_INTO_STAND_FRAMES  # 250 - 60 = 190

DATASET_PATH         = pathlib.Path('/kaggle/input/datasets/lucyfirst/beef-cattle-behavior-data-set/Category Videos/cows/')
DRINK_DATASET_PATH   = pathlib.Path('/kaggle/input/datasets/muhammadharris03/drink-v03/drink')
CLEANED_DATASET_PATH = pathlib.Path('/kaggle/working/cleaned_dataset_v07/')

ALL_LABELS     = ['drink', 'eat', 'lie', 'stand']
AUGMENT_LABELS = ['drink', 'eat']

# AUGMENTATION FUNCTIONS

In [17]:
def crop_head_contact_zone(frame: np.ndarray) -> np.ndarray:
    h, w = frame.shape[:2]
    return cv2.resize(frame[int(h * 0.30):, :], (w, h))

def denoise_frame(frame: np.ndarray) -> np.ndarray:
    return cv2.fastNlMeansDenoisingColored(frame, None, 10, 10, 7, 21)

def stretch_contrast(frame: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_stretched = cv2.normalize(l, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(cv2.merge([l_stretched, a, b]), cv2.COLOR_LAB2BGR)

def enhance_contact_zone(frame: np.ndarray) -> np.ndarray:
    h = frame.shape[0]
    split  = int(h * 0.60)
    top    = frame[:split, :]
    bottom = frame[split:, :]
    lab    = cv2.cvtColor(bottom, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(4, 4))
    bottom = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)
    return np.vstack([top, bottom])

def sharpen_frame(frame: np.ndarray) -> np.ndarray:
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    return cv2.filter2D(frame, -1, kernel)

def mouth_contact_preprocessing(frame: np.ndarray) -> np.ndarray:
    frame = crop_head_contact_zone(frame)
    frame = denoise_frame(frame)
    frame = stretch_contrast(frame)
    frame = enhance_contact_zone(frame)
    frame = sharpen_frame(frame)
    return frame

def augment_variant(frame: np.ndarray, variant: int) -> np.ndarray:
    frame = mouth_contact_preprocessing(frame)
    if variant == 1:
        frame = cv2.flip(frame, 1)
    elif variant == 2:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * 1.2, 0, 255)
        frame = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    elif variant == 3:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * 0.8, 0, 255)
        frame = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    elif variant == 4:
        h, w = frame.shape[:2]
        cy, cx = int(h * 0.10), int(w * 0.10)
        frame = cv2.resize(frame[cy:h-cy, cx:w-cx], (w, h))
    return frame

# HELPER FUNCTIONS

In [19]:
def recreate_dir(dir_path: pathlib.Path) -> None:
    if dir_path.exists():
        shutil.rmtree(dir_path)
    dir_path.mkdir(parents=True, exist_ok=True)


def oversample_images_to_target(
    source_frames: list,
    output_folder: str,
    target: int,
    start_count: int = 0,
    label_tag: str = '',
    use_augmentation: bool = True
) -> int:
    
    saved   = start_count
    variant = 0

    while saved < target:
        random.shuffle(source_frames)
        for fp in source_frames:
            if saved >= target:
                break
            frame = cv2.imread(str(fp))
            if frame is None:
                continue
            processed = augment_variant(frame, variant % 5) if use_augmentation else frame
            out_name  = f"{pathlib.Path(fp).stem}_aug{variant}_{saved:04d}.jpg"
            cv2.imwrite(os.path.join(output_folder, out_name), processed)
            saved += 1
        variant += 1

    print(f"  [{label_tag}] → {saved} frames (target: {target})")
    return saved


def extract_frames_from_video(
    video_path: str,
    output_folder: str,
    interval: int,
    current_count: int,
    cap_limit: int,
    apply_augmentation: bool = False
) -> int:
    
    if current_count >= cap_limit:
        return current_count

    cap        = cv2.VideoCapture(str(video_path))
    frame_idx  = 0
    video_stem = pathlib.Path(video_path).stem

    while True:
        ret, frame = cap.read()
        if not ret or current_count >= cap_limit:
            break
        if frame_idx % interval == 0:
            fname     = os.path.join(output_folder, f"{video_stem}_f{current_count:04d}.jpg")
            processed = mouth_contact_preprocessing(frame) if apply_augmentation else frame
            cv2.imwrite(fname, processed)
            current_count += 1
        frame_idx += 1

    cap.release()
    return current_count

# EXECUTION

In [ ]:
# Reset output folders (safe to rerun)
for split in ['train', 'val', 'test']:
    for cls in ALL_LABELS:
        recreate_dir(CLEANED_DATASET_PATH / split / cls)

# DRINK: augmentation ON
print("=" * 55)
print("PROCESSING: DRINK")
print("=" * 55)

all_drink = sorted([
    DRINK_DATASET_PATH / f
    for f in os.listdir(DRINK_DATASET_PATH)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])
random.shuffle(all_drink)
print(f"Found {len(all_drink)} pre-extracted drink frames")

d_train_end = int(len(all_drink) * 0.80)
d_val_end   = int(len(all_drink) * 0.90)
drink_train = all_drink[:d_train_end]
drink_val   = all_drink[d_train_end:d_val_end]
drink_test  = all_drink[d_val_end:]

print(f"Source — Train: {len(drink_train)} | Val: {len(drink_val)} | Test: {len(drink_test)}")

oversample_images_to_target(
    drink_train,
    str(CLEANED_DATASET_PATH / 'train' / 'drink'),
    MAX_TRAIN_FRAMES,
    label_tag='drink train'
)
for fp in tqdm.tqdm(drink_val, desc="Drink Val"):
    frame = cv2.imread(str(fp))
    cv2.imwrite(str(CLEANED_DATASET_PATH / 'val' / 'drink' / fp.name),
                mouth_contact_preprocessing(frame))
for fp in tqdm.tqdm(drink_test, desc="Drink Test"):
    frame = cv2.imread(str(fp))
    cv2.imwrite(str(CLEANED_DATASET_PATH / 'test' / 'drink' / fp.name),
                mouth_contact_preprocessing(frame))



# EAT: augmentation ON
print("\n" + "=" * 55)
print("PROCESSING: EAT (eat videos only)")
print("=" * 55)

eat_video_dir   = DATASET_PATH / 'eat'
eat_video_paths = sorted([
    eat_video_dir / f
    for f in os.listdir(eat_video_dir)
    if f.lower().endswith(('.mp4', '.avi'))
])
random.shuffle(eat_video_paths)
print(f"Found {len(eat_video_paths)} eat videos")

eat_train_vids = eat_video_paths[:-2]
eat_val_vid    = eat_video_paths[-2]
eat_test_vid   = eat_video_paths[-1]

total_eat = 0
for path in tqdm.tqdm(eat_train_vids, desc="Eat Train"):
    total_eat = extract_frames_from_video(
        str(path),
        str(CLEANED_DATASET_PATH / 'train' / 'eat'),
        FRAME_EXTRACTION_INTERVAL,
        total_eat,
        MAX_TRAIN_FRAMES,
        apply_augmentation=True
    )
    if total_eat >= MAX_TRAIN_FRAMES:
        break

if total_eat < MAX_TRAIN_FRAMES:
    print(f"  Eat videos gave {total_eat} frames, oversampling to {MAX_TRAIN_FRAMES}...")
    existing = list((CLEANED_DATASET_PATH / 'train' / 'eat').glob('*.jpg'))
    total_eat = oversample_images_to_target(
        existing,
        str(CLEANED_DATASET_PATH / 'train' / 'eat'),
        MAX_TRAIN_FRAMES,
        start_count=total_eat,
        label_tag='eat oversample',
        use_augmentation=True
    )

print(f"  Eat train done: {total_eat} frames")

extract_frames_from_video(str(eat_val_vid),
    str(CLEANED_DATASET_PATH / 'val' / 'eat'),
    FRAME_EXTRACTION_INTERVAL, 0, VAL_FRAMES, apply_augmentation=True)

extract_frames_from_video(str(eat_test_vid),
    str(CLEANED_DATASET_PATH / 'test' / 'eat'),
    FRAME_EXTRACTION_INTERVAL, 0, TEST_FRAMES, apply_augmentation=True)


# LIE: no augmentation
print("\n" + "=" * 55)
print("PROCESSING: LIE (raw frames)")
print("=" * 55)

lie_video_dir   = DATASET_PATH / 'lie'
lie_video_paths = sorted([
    lie_video_dir / f
    for f in os.listdir(lie_video_dir)
    if f.lower().endswith(('.mp4', '.avi'))
])
random.shuffle(lie_video_paths)
print(f"Found {len(lie_video_paths)} lie videos")

lie_train_vids = lie_video_paths[:-2]
lie_val_vid    = lie_video_paths[-2] if len(lie_video_paths) >= 2 else lie_video_paths[0]
lie_test_vid   = lie_video_paths[-1]

total_lie = 0
for path in tqdm.tqdm(lie_train_vids, desc="Lie Train"):
    total_lie = extract_frames_from_video(
        str(path),
        str(CLEANED_DATASET_PATH / 'train' / 'lie'),
        FRAME_EXTRACTION_INTERVAL,
        total_lie, MAX_TRAIN_FRAMES,
        apply_augmentation=False
    )
    if total_lie >= MAX_TRAIN_FRAMES:
        break

if total_lie < MAX_TRAIN_FRAMES:
    print(f"  Lie got {total_lie} frames, oversampling to {MAX_TRAIN_FRAMES}...")
    existing = list((CLEANED_DATASET_PATH / 'train' / 'lie').glob('*.jpg'))
    oversample_images_to_target(existing,
        str(CLEANED_DATASET_PATH / 'train' / 'lie'),
        MAX_TRAIN_FRAMES, start_count=total_lie,
        label_tag='lie train', use_augmentation=False)

extract_frames_from_video(str(lie_val_vid),
    str(CLEANED_DATASET_PATH / 'val' / 'lie'),
    FRAME_EXTRACTION_INTERVAL, 0, VAL_FRAMES, apply_augmentation=False)

extract_frames_from_video(str(lie_test_vid),
    str(CLEANED_DATASET_PATH / 'test' / 'lie'),
    FRAME_EXTRACTION_INTERVAL, 0, TEST_FRAMES, apply_augmentation=False)


# STAND + RUMINATE
print("\n" + "=" * 55)
print("PROCESSING: STAND (stand videos + ruminate mix, raw frames)")
print("=" * 55)

stand_video_dir   = DATASET_PATH / 'stand'
stand_video_paths = sorted([
    stand_video_dir / f
    for f in os.listdir(stand_video_dir)
    if f.lower().endswith(('.mp4', '.avi'))
])
random.shuffle(stand_video_paths)
print(f"Found {len(stand_video_paths)} stand videos")

stand_train_vids = stand_video_paths[:-2]
stand_val_vid    = stand_video_paths[-2] if len(stand_video_paths) >= 2 else stand_video_paths[0]
stand_test_vid   = stand_video_paths[-1]

total_stand = 0
for path in tqdm.tqdm(stand_train_vids, desc="Stand Train"):
    total_stand = extract_frames_from_video(
        str(path),
        str(CLEANED_DATASET_PATH / 'train' / 'stand'),
        FRAME_EXTRACTION_INTERVAL,
        total_stand, STAND_OWN_FRAMES,
        apply_augmentation=False
    )
    if total_stand >= STAND_OWN_FRAMES:
        break

if total_stand < STAND_OWN_FRAMES:
    print(f"  Stand videos gave {total_stand}, oversampling to {STAND_OWN_FRAMES}...")
    existing = list((CLEANED_DATASET_PATH / 'train' / 'stand').glob('*.jpg'))
    total_stand = oversample_images_to_target(existing,
        str(CLEANED_DATASET_PATH / 'train' / 'stand'),
        STAND_OWN_FRAMES, start_count=total_stand,
        label_tag='stand oversample', use_augmentation=False)

print(f"  Stand own frames: {total_stand}")

print(f"  Adding {RUMINATE_INTO_STAND_FRAMES} ruminate frames into stand...")
ruminate_dir    = DATASET_PATH / 'ruminate'
ruminate_videos = sorted([
    ruminate_dir / f
    for f in os.listdir(ruminate_dir)
    if f.lower().endswith(('.mp4', '.avi'))
])
random.shuffle(ruminate_videos)

total_stand_combined = total_stand
for path in tqdm.tqdm(ruminate_videos[:-1], desc="Ruminate → Stand"):
    total_stand_combined = extract_frames_from_video(
        str(path),
        str(CLEANED_DATASET_PATH / 'train' / 'stand'),
        FRAME_EXTRACTION_INTERVAL,
        total_stand_combined,
        MAX_TRAIN_FRAMES,       
        apply_augmentation=False  
    )
    if total_stand_combined >= MAX_TRAIN_FRAMES:
        break

current_stand = len(list((CLEANED_DATASET_PATH / 'train' / 'stand').glob('*.jpg')))
if current_stand < MAX_TRAIN_FRAMES:
    print(f"  Stand at {current_stand}, final oversample to {MAX_TRAIN_FRAMES}...")
    existing = list((CLEANED_DATASET_PATH / 'train' / 'stand').glob('*.jpg'))
    oversample_images_to_target(existing,
        str(CLEANED_DATASET_PATH / 'train' / 'stand'),
        MAX_TRAIN_FRAMES, start_count=current_stand,
        label_tag='stand final', use_augmentation=False)

extract_frames_from_video(str(stand_val_vid),
    str(CLEANED_DATASET_PATH / 'val' / 'stand'),
    FRAME_EXTRACTION_INTERVAL, 0, VAL_FRAMES, apply_augmentation=False)

extract_frames_from_video(str(stand_test_vid),
    str(CLEANED_DATASET_PATH / 'test' / 'stand'),
    FRAME_EXTRACTION_INTERVAL, 0, TEST_FRAMES, apply_augmentation=False)

PROCESSING: DRINK
Found 303 pre-extracted drink frames
Source — Train: 242 | Val: 30 | Test: 31
  [drink train] → 250 frames (target: 250)


Drink Test: 100%|██████████| 31/31 [00:04<00:00,  6.79it/s]



PROCESSING: EAT (eat videos only)
Found 546 eat videos


Eat Train:   4%|▍         | 24/544 [00:37<13:29,  1.56s/it]


  Eat train done: 250 frames

PROCESSING: LIE (raw frames)
Found 1362 lie videos


...


PROCESSING: STAND (stand videos + ruminate mix, raw frames)
Found 638 stand videos


Stand Train:   3%|▎         | 18/636 [00:01<00:46, 13.42it/s]


  Stand own frames: 190
  Adding 60 ruminate frames into stand...


Ruminate → Stand:   0%|          | 5/1543 [00:00<01:36, 15.93it/s]


10

# FINAL SUMMARY

In [21]:
print(f"\n{'=' * 55}")
print(f"✅ Done! Dataset: {CLEANED_DATASET_PATH}")
print(f"{'=' * 55}\n")

all_good = True
for split in ['train', 'val', 'test']:
    target = MAX_TRAIN_FRAMES if split == 'train' else (VAL_FRAMES if split == 'val' else TEST_FRAMES)
    print(f"  {split.upper()} (target per label: {target}):")
    for cls in ALL_LABELS:
        path   = CLEANED_DATASET_PATH / split / cls
        count  = len(list(path.glob('*.jpg'))) if path.exists() else 0
        if cls == 'stand':
            note = '(raw + ruminate mix)'
        elif cls == 'eat':
            note = '(augmented, eat only)'
        elif cls == 'drink':
            note = '(augmented)'
        else:
            note = '(raw)'
        status = '✅' if count > 0 else '❌ EMPTY'
        if count == 0:
            all_good = False
        print(f"    {cls:<10}: {count:>4} frames {note} {status}")
    print()

if all_good:
    print("✅ All labels populated. Update training:")
    print("   data='/kaggle/working/cleaned_dataset_v07'")
else:
    print("❌ Fix empty labels before training.")


✅ Done! Dataset: /kaggle/working/cleaned_dataset_v07

  TRAIN (target per label: 250):
    drink     :  629 frames (augmented) ✅
    eat       :  500 frames (augmented, eat only) ✅
    lie       :  500 frames (raw) ✅
    stand     :  500 frames (raw + ruminate mix) ✅

  VAL (target per label: 25):
    drink     :   58 frames (augmented) ✅
    eat       :   20 frames (augmented, eat only) ✅
    lie       :   20 frames (raw) ✅
    stand     :   20 frames (raw + ruminate mix) ✅

  TEST (target per label: 25):
    drink     :   58 frames (augmented) ✅
    eat       :   20 frames (augmented, eat only) ✅
    lie       :   20 frames (raw) ✅
    stand     :   20 frames (raw + ruminate mix) ✅

✅ All labels populated. Update training:
   data='/kaggle/working/cleaned_dataset_v07'


# Model Training

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

model = YOLO('yolo26n-cls.pt')

results = model.train(
    data="/kaggle/working/cow_behavior.yaml",
    epochs=50,
    patience=10,
    imgsz=224,
    batch=32,
    optimizer='Adam',
    lr0=0.00025,      
    dropout=0.4,     
    cache='disk',
    workers=4,
    mixup=0.3, 
    name='yolo_cow_v5_5label_26n'
)
model.info()

metrics = model.val()
print(f"Top-1 Accuracy: {metrics.top1:.3f}")   
print(f"Top-5 Accuracy: {metrics.top5:.3f}")   

# Testing the Model

# Imports

In [ ]:
import cv2
import pathlib
import numpy as np
!pip install ultralytics
from ultralytics import YOLO

# CONFIGURATION

In [ ]:

MODEL_PATH  = '/kaggle/input/models/muhammadharris03/model-aug-26n/pytorch/default/1/best_5class_aug_26n.pt'
IMG_SIZE    = 224
CLASS_NAMES = ['drink', 'eat', 'lie', 'stand']  

SAMPLE_EVERY = 15

model = YOLO(MODEL_PATH)
CLASS_NAMES = list(model.names.values())
print(f"✅ Model loaded | Classes: {CLASS_NAMES}")

# PREPROCESSING

In [ ]:
def preprocess(frame: np.ndarray) -> np.ndarray:
    frame = crop_head_contact_zone(frame)
    frame = denoise_frame(frame)
    frame = stretch_contrast(frame)
    frame = enhance_contact_zone(frame)
    frame = sharpen_frame(frame)
    frame = edge_enhance_bottom(frame)
    return frame

# PREDICTION

- Images

In [ ]:
def predict_image(image_path: str) -> dict:    
    frame = cv2.imread(image_path)
    if frame is None:
        return {'error': f'Could not read image: {image_path}'}


    processed = preprocess(frame)

    tmp_path = '/tmp/_pred_frame.jpg'
    cv2.imwrite(tmp_path, processed)

    results    = model(tmp_path, imgsz=IMG_SIZE, verbose=False)
    probs      = results[0].probs
    predicted  = CLASS_NAMES[probs.top1]
    confidence = float(probs.top1conf) * 100

    return {
        'type'      : 'image',
        'input'     : image_path,
        'predicted' : predicted,
        'confidence': f"{confidence:.1f}%",
        'all_probs' : {
            CLASS_NAMES[i]: f"{float(probs.data[i]) * 100:.1f}%"
            for i in range(len(CLASS_NAMES))
        }
    }

- Videos

In [ ]:
def predict_video(video_path: str) -> dict:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {'error': f'Could not open video: {video_path}'}

    predictions = []
    confidences = {}
    frame_idx   = 0
    tmp_path    = '/tmp/_pred_frame.jpg'

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % SAMPLE_EVERY == 0:
            processed = preprocess(frame)
            cv2.imwrite(tmp_path, processed)

            results   = model(tmp_path, imgsz=IMG_SIZE, verbose=False)
            probs     = results[0].probs
            pred      = CLASS_NAMES[probs.top1]
            conf      = float(probs.top1conf)

            predictions.append(pred)
            if pred not in confidences:
                confidences[pred] = []
            confidences[pred].append(conf)

        frame_idx += 1

    cap.release()

    if not predictions:
        return {'error': 'No frames could be read from video'}

    final_pred = max(set(predictions), key=predictions.count)
    avg_conf   = np.mean(confidences[final_pred]) * 100

    return {
        'type'          : 'video',
        'input'         : video_path,
        'predicted'     : final_pred,
        'confidence'    : f"{avg_conf:.1f}%",
        'frame_votes'   : {c: predictions.count(c) for c in set(predictions)},
        'frames_sampled': len(predictions)
    }

- PREDICTOR

In [ ]:
def predict(input_path: str) -> dict:
    ext = pathlib.Path(input_path).suffix.lower()

    if ext in ['.jpg', '.jpeg', '.png']:
        result = predict_image(input_path)

    elif ext in ['.mp4', '.avi', '.mov']:
        result = predict_video(input_path)

    else:
        print(f"❌ Unsupported file type: {ext}")
        return {}

    print("\n" + "=" * 45)
    print(f"  Input     : {pathlib.Path(result['input']).name}")
    print(f"  Type      : {result['type']}")
    print(f"  Predicted : {result['predicted'].upper()}")
    print(f"  Confidence: {result['confidence']}")

    if result['type'] == 'video':
        print(f"  Frame votes    : {result['frame_votes']}")
        print(f"  Frames sampled : {result['frames_sampled']}")

    if result['type'] == 'image':
        print(f"  All probs :")
        for cls, prob in result['all_probs'].items():
            bar = '█' * int(float(prob.strip('%')) / 5)
            print(f"    {cls:<12}: {prob:>7}  {bar}")

    print("=" * 45)
    return result

- USAGE — EDIT BELOW THIS LINE

In [ ]:
predict('/kaggle/input/datasets/muhammadharris03/newdata-img/cow1160.jpg')